# Practical — LightGBM on Census Income (Adult) 🧀🧀🧀

**One library, three ensembles.** You will use **LightGBM** as a single lens on the
whole trees chapter: the *same* library gives you a single decision tree, a random
forest, and gradient boosting — so the only thing that changes between them is the
**ensembling strategy**, not the implementation, the preprocessing, or the metric.

**Dataset — UCI Adult / "Census Income".** Each row is one **adult** from the **1994 US
Census** database (extracted by Barry Becker & Ronny Kohavi); predict whether their income is
**> \$50K/year** from ~14 features — **demographics** (age, education, marital status,
occupation, relationship, race, sex, native country) and **work / finance** (workclass,
hours-per-week, capital gain/loss). It is **imbalanced** (~24% earn >\$50K), with
**categorical** features and **missing values** — a realistic tabular problem, not a toy. It is
also *the* canonical benchmark in **algorithmic-fairness** research (hence the sex slice-check
in Part 3).

Source: [UCI](https://archive.ics.uci.edu/dataset/2/adult) · [OpenML id 1590](https://www.openml.org/d/1590).

**Metric rule:** ~76% of people earn ≤50K, so *accuracy is misleading* (predicting
`<=50K` for everyone already scores 0.76). **Report ROC-AUC and F1 throughout.**

This is a **minimal-scaffold** project: you do the cleaning and write all the modelling
code. Only the data loader is given.

## Setup

Needs `lightgbm` and `scikit-learn` (already in the course `ma` venv). If missing:
`uv pip install lightgbm scikit-learn`.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml

# --- the ONLY code you are given: the loader ---
adult = fetch_openml("adult", version=2, as_frame=True)
df = adult.frame          # 48842 rows x 15 cols; target column is 'class'
print(df.shape)
df.head()

The target is the column **`class`** with values `>50K` (positive) / `<=50K`.
Everything below is up to you.

## Part 0 — Clean and explore the data (do it yourself)

1. **Target.** Make a binary label `y = (class == '>50K')`. What fraction is positive?
   What accuracy does the trivial "always predict `<=50K`" baseline get? (This is why we
   use ROC-AUC / F1, not accuracy.)
2. **Missing values.** Which columns have them, and how much? (Hint: `workclass`,
   `occupation`, `native-country`.) You do **not** have to impute — LightGBM handles
   `NaN` natively. Decide and justify.
3. **Redundant / non-predictive columns.** Drop **`fnlwgt`** (it is a census *sampling
   weight*, not a real predictor). Notice that **`education`** and **`education-num`**
   carry the same information (one is the ordinal code of the other) — keep one.
4. **Categoricals.** List which columns are categorical (there are 8, including
   `native-country` with **41** levels). You will need this list for LightGBM's
   `categorical_feature=` argument. Make sure they are pandas `category` dtype.
5. **Split.** Make a **stratified** train / test split and *reuse the same split*
   everywhere below, so all models are compared on identical data.

In [ ]:
# TODO Part 0: build y, inspect imbalance + missingness, drop fnlwgt / dup education,
#              mark categoricals, make one stratified train/test split.

## Part 1 — Baselines: one library (+ a linear reference)

First a **non-tree reference** to beat: **logistic regression**. Unlike the trees, it cannot
eat the raw frame — it needs **imputation + one-hot + scaling** (a `ColumnTransformer` +
`LogisticRegression` pipeline). Fit it and put it on the **scoreboard** (ROC-AUC + F1): it is
the yardstick for "did the trees actually earn their complexity?"

Then fit the three ensembles **through LightGBM** on your Part-0 split (the *raw* frame — NaN
and categoricals handled natively), scored on the same **test set**. Keep the running
**scoreboard** (a dict or DataFrame).

- **Single decision tree** = `LGBMClassifier(n_estimators=1, learning_rate=1.0)` — the
  "atom" from [17]. (Read it; note it is unstable.)
- **Random forest** = `LGBMClassifier(boosting_type='rf', bagging_fraction=0.8,`
  `bagging_freq=1, feature_fraction=0.8, n_estimators=300)` — the variance-cut from [18].
- **Gradient boosting** = `LGBMClassifier()` (default `gbdt`) — the bias-cut from [19]/[20].

Then the **categorical question**: fit the GBDT twice — once passing
`categorical_feature=` (native handling) and once on a **one-hot** encoded frame
(`pd.get_dummies`). How many columns does one-hot create (look at `native-country`)?
Does native handling change ROC-AUC and/or training time?

**Expected:** logreg ≈ single tree < RF ≲ GBDT. Report it honestly even if RF is close.

In [ ]:
# TODO Part 1:
#   - logistic-regression baseline: ColumnTransformer(median-impute + scale numerics,
#       most_frequent-impute + one-hot categoricals) -> LogisticRegression; AUC + F1 -> scoreboard.
#   - single tree / rf / gbdt via LightGBM on the RAW frame; AUC + F1 -> scoreboard.
#   - native categorical vs one-hot GBDT comparison (AUC + column count + fit time).

## Part 2 — Make the GBDT win (tuning)

1. **Early stopping.** Hold out a validation set, set `n_estimators` large, and stop when
   validation AUC stops improving (`from lightgbm import early_stopping`,
   `callbacks=[early_stopping(50)]`, `eval_metric="auc"`). Early stopping picks the number
   of trees for you.
2. **Tune in the [20] order** (by hand first, to see *why* each knob matters):
   `learning_rate` ↔ `n_estimators`, then `num_leaves` / `min_data_in_leaf`, then
   `lambda_l1` / `lambda_l2` / `min_gain_to_split`, then subsampling
   (`bagging_fraction`, `feature_fraction`).
3. **Then automate it with a Bayesian search — Optuna** (already in the `ma` env). Write an
   `objective(trial)` that samples those hyperparameters (`trial.suggest_float/int`), fits with
   early stopping, and returns validation AUC; run
   `optuna.create_study(direction="maximize").optimize(objective, n_trials=40)` and compare its
   best to your hand-tuned result.
4. **Imbalance.** Try `is_unbalance=True` or `scale_pos_weight`, and pick the **decision
   threshold** by F1 (not the default 0.5).
5. Beat your Part-1 GBDT on **ROC-AUC** and add the tuned model to the scoreboard.

In [ ]:
# TODO Part 2: early stopping; hand-tune in the [20] order; then an Optuna (TPE) study over
#              the same space; imbalance handling (scale_pos_weight); F1-based threshold.

## Part 3 — Read the model

1. **Gain importance** (`model.feature_importances_`) — and its bias toward
   high-cardinality features (which columns look suspiciously important?).
2. **Permutation importance** (`sklearn.inspection.permutation_importance`) on the test
   set — compare to the gain ranking. Which do you trust more, and why?
3. *Optional (non-linearity detector):* compute **permutation importance for your logreg
   baseline and your GBDT** (same `scoring='roc_auc'`). A large **GBDT minus logreg** gap on a
   **numeric** feature flags a non-linear effect the line cannot fit; confirm with a
   **partial-dependence plot** (try `age`, `capital-gain`). Categoricals do not count here —
   one-hot logreg is already flexible per level.
4. *Optional:* add a **monotonic constraint** (`monotone_constraints`) forcing
   `education-num` to have a **non-decreasing** effect on P(>50K). Does test AUC hold up?
5. **Subgroup slice-check.** Split the test set by **`sex`** and report your tuned model'"'"'s
   **ROC-AUC and predicted-positive rate** within each group. A strong overall AUC can hide
   very different behavior per group: does the model rank as well for both, and does it
   predict `>50K` far more often for one? Write one or two sentences on what you find.
6. **Reflection (write a short paragraph):** did the tree < RF < GBDT ordering hold, and by
   how much? Where did tuning help most? What would you try next?

In [ ]:
# TODO Part 3: gain vs permutation importance; optional monotonic constraint;
#              slice-check (ROC-AUC + positive rate) by sex; reflection.

## What to submit

- the **scoreboard** (logreg / single tree / RF / GBDT / tuned GBDT — ROC-AUC + F1);
- the native-categorical **vs** one-hot comparison (AUC + column count + time);
- your **tuned** model's AUC/F1 and the tuning you did;
- the **importance** comparison (gain vs permutation);
- the **slice-check** by `sex` (per-group AUC + positive rate) and your read of it;
- your written **reflection**.